# GreenML GBDT Benchmark — Revised v3
### Addresses reviewer feedback: 25 repeats, BF16, statistical tests, fixed seeds, CI

**Key changes from v2:**
- N_REPEATS = 25 (was 5)
- Added BF16 as 4th precision profile
- Fixed random seeds everywhere (deterministic)
- Batch sizes: 100, 1000, 10000, 100000 (dropped 1 and 10 — noise-dominated)
- Wilcoxon signed-rank tests + 95% CI computed automatically
- CodeCarbon RAPL domain logged
- Checkpoint saving after each dataset — safe for free Colab tier
- Full reproducibility table generated

**Estimated runtime: ~5-6 hours on Colab free CPU**
**Hardware confirmed: Intel Xeon Broadwell (model 79), AVX2, f16c, no AVX-512**

## 0. Install & Mount

In [ ]:
!pip install codecarbon lightgbm xgboost catboost ml_dtypes --quiet

from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted.')

## 1. Reproducibility & Hardware Fingerprint
Run this cell first. Output goes directly into the paper's reproducibility table.

In [ ]:
import subprocess, platform, json
from pathlib import Path
import lightgbm, xgboost, catboost, codecarbon, sklearn, numpy, pandas

# ── Package versions ──────────────────────────────────────────────────────────
versions = {
    'Python'    : platform.python_version(),
    'OS'        : platform.platform(),
    'LightGBM'  : lightgbm.__version__,
    'XGBoost'   : xgboost.__version__,
    'CatBoost'  : catboost.__version__,
    'CodeCarbon': codecarbon.__version__,
    'scikit-learn': sklearn.__version__,
    'NumPy'     : numpy.__version__,
    'pandas'    : pandas.__version__,
}

# ── CPU details ───────────────────────────────────────────────────────────────
cpuinfo = open('/proc/cpuinfo').read()
cpu_name = [l for l in cpuinfo.split('\n') if 'model name' in l][0].split(':')[1].strip()
cpu_flags = [l for l in cpuinfo.split('\n') if l.startswith('flags')][0].split(':')[1].strip().split()

ISA_FLAGS = ['avx', 'avx2', 'avx512f', 'avx512_fp16', 'avx512_bf16', 'f16c', 'fma', 'sse4_2']
isa = {f: (f in cpu_flags) for f in ISA_FLAGS}

# ── RAPL domain check ─────────────────────────────────────────────────────────
rapl_domains = []
try:
    rapl_path = Path('/sys/class/powercap/intel-rapl')
    if rapl_path.exists():
        for p in rapl_path.rglob('name'):
            rapl_domains.append(p.read_text().strip())
except Exception:
    rapl_domains = ['RAPL not directly accessible (virtualised environment)']

# ── Print report ──────────────────────────────────────────────────────────────
print('='*60)
print('HARDWARE & SOFTWARE FINGERPRINT')
print('='*60)
print(f'CPU Model : {cpu_name}')
print(f'CPU Family: Intel Broadwell (model 79), ~2016')
print()
print('ISA / Instruction Set Support:')
for f, v in isa.items():
    status = '✓ YES' if v else '✗ NO '
    note = ''
    if f == 'f16c':   note = '← FP16↔FP32 conversion ONLY (no native FP16 arithmetic)'
    if f == 'avx512_fp16': note = '← Required for native FP16 arithmetic (absent here)'
    if f == 'avx2':   note = '← 256-bit SIMD: 8×FP32 or 4×FP64 per lane'
    print(f'  {f:<20}: {status}  {note}')
print()
print('FP16 Arithmetic Behaviour on This CPU:')
print('  Every FP16 op = VCVTPH2PS (→FP32) + FP32_op + VCVTPS2PH (→FP16)')
print('  = 3 instructions per 1 FP16 arithmetic operation')
print('  = higher energy than native FP32 (1 instruction)')
print('  Native FP16 requires avx512_fp16 (Intel Sapphire Rapids, 2023+)')
print()
print('RAPL Domains accessible:', rapl_domains if rapl_domains else 'None (virtualised)')
print('CodeCarbon fallback: estimates via CPU TDP heuristic in virtualised environments')
print()
print('Package Versions:')
for k, v in versions.items():
    print(f'  {k:<15}: {v}')

# Save for paper
BASE = Path('/content/drive/MyDrive/MyResearchWork/phdconf')
RESULTS = BASE / 'results_v3'
(RESULTS / 'figures').mkdir(parents=True, exist_ok=True)
(RESULTS / 'tables').mkdir(parents=True, exist_ok=True)
(RESULTS / 'stats').mkdir(parents=True, exist_ok=True)

fingerprint = {'versions': versions, 'cpu': cpu_name, 'isa': isa, 'rapl': rapl_domains}
json.dump(fingerprint, open(RESULTS / 'hardware_fingerprint.json', 'w'), indent=2)
print('\n✅ Fingerprint saved to Drive.')

## 2. Imports & Global Config

In [ ]:
import time, warnings, gc, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from contextlib import contextmanager
from dataclasses import dataclass
from typing import List
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from codecarbon import EmissionsTracker

# ── BF16 support via ml_dtypes ────────────────────────────────────────────────
try:
    import ml_dtypes
    BF16_DTYPE = ml_dtypes.bfloat16
    BF16_AVAILABLE = True
    print('✅ BF16 available via ml_dtypes')
except ImportError:
    BF16_AVAILABLE = False
    print('⚠ ml_dtypes not available — BF16 will be skipped')

warnings.filterwarnings('ignore')

# ── FIXED SEEDS — critical for reproducibility and AUC consistency ────────────
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE      = Path('/content/drive/MyDrive/MyResearchWork/phdconf')
DATA_IEEE = BASE / 'datasets' / 'IEEE-CIS'
DATA_HM   = BASE / 'datasets' / 'HandM'
RESULTS   = BASE / 'results_v3'

# ── Precision profiles ────────────────────────────────────────────────────────
PRECISION_PROFILES = {'FP64': np.float64, 'FP32': np.float32, 'FP16': np.float16}
if BF16_AVAILABLE:
    PRECISION_PROFILES['BF16'] = BF16_DTYPE

# ── Experiment config ─────────────────────────────────────────────────────────
# Dropped batch sizes 1 and 10 — noise-dominated at this measurement granularity
BATCH_SIZES = [100, 1_000, 10_000, 100_000]
N_REPEATS   = 25   # was 5 — increased for statistical validity
N_WARMUP    = 1    # discarded runs

print(f'\nPrecision profiles : {list(PRECISION_PROFILES.keys())}')
print(f'Batch sizes        : {BATCH_SIZES}')
print(f'Repeats per config : {N_REPEATS} (+ {N_WARMUP} warm-up discarded)')
total = 2 * 3 * len(PRECISION_PROFILES) * len(BATCH_SIZES) * 3 * N_REPEATS
print(f'Total measurements : {total:,}')
print('\n✅ Config ready.')

## 3. Energy Measurement Utilities

In [ ]:
@dataclass
class StageMetrics:
    stage: str; precision: str; batch_size: int; model: str; dataset: str
    latency_ms: float = 0.0
    energy_mj:  float = 0.0
    co2e_ug:    float = 0.0
    auc:        float = 0.0
    score_drift: float = 0.0


@contextmanager
def measure_stage(stage, precision, batch_size, model_name, dataset):
    m = StageMetrics(stage, precision, batch_size, model_name, dataset)
    tracker = EmissionsTracker(
        project_name=f'{model_name}_{stage}_{precision}_{batch_size}',
        output_dir=str(RESULTS),
        log_level='error',
        save_to_file=False,
        tracking_mode='process',
        measure_power_secs=0.5,
    )
    tracker.start()
    t0 = time.perf_counter()
    yield m
    t1 = time.perf_counter()
    emissions_kg = tracker.stop() or 0.0
    m.latency_ms = (t1 - t0) * 1000
    try:
        energy_kwh = tracker._total_energy.kWh
    except Exception:
        # Fallback: estimate from emissions using standard grid factor
        # CodeCarbon uses CPU TDP heuristic in virtualised environments
        # where RAPL is inaccessible. This is disclosed in methodology.
        energy_kwh = emissions_kg / 0.233
    m.energy_mj = energy_kwh * 3.6e9   # 1 kWh = 3.6×10^9 mJ
    m.co2e_ug   = emissions_kg * 1e9   # kg → µg


print('✅ Measurement utilities ready.')

## 4. Dataset Loading

In [ ]:
def reduce_mem(df):
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype(np.float32)
    for col in df.select_dtypes('int64').columns:
        df[col] = df[col].astype(np.int32)
    return df


# ── IEEE-CIS ──────────────────────────────────────────────────────────────────
print('Loading IEEE-CIS...', flush=True)
trans = pd.read_csv(DATA_IEEE / 'train_transaction.csv', low_memory=False)
ident = pd.read_csv(DATA_IEEE / 'train_identity.csv',   low_memory=False)
trans = reduce_mem(trans)
ident = reduce_mem(ident)
df_ieee = trans.merge(ident, on='TransactionID', how='left')
del trans, ident; gc.collect()

null_pct     = df_ieee.isnull().mean()
feature_cols = [c for c in df_ieee.columns
                if c not in ['TransactionID', 'isFraud']
                and null_pct[c] <= 0.60]

CAT_COLS = ['ProductCD','card4','card6','P_emaildomain','R_emaildomain',
            'M1','M2','M3','M4','M5','M6','M7','M8','M9',
            'id_12','id_15','id_16','id_23','id_27','id_28','id_29',
            'id_30','id_31','id_33','id_34','id_35','id_36','id_37','id_38',
            'DeviceType','DeviceInfo']
CAT_COLS = [c for c in CAT_COLS if c in feature_cols]

for col in CAT_COLS:
    df_ieee[col] = df_ieee[col].astype(str).fillna('missing')
    df_ieee[col] = LabelEncoder().fit_transform(df_ieee[col]).astype(np.int32)

X_ieee = df_ieee[feature_cols].fillna(-1).astype(np.float32)
y_ieee = df_ieee['isFraud'].astype(np.int32)
del df_ieee; gc.collect()

X_tr_ieee, X_te_ieee, y_tr_ieee, y_te_ieee = train_test_split(
    X_ieee, y_ieee, test_size=0.2, stratify=y_ieee, random_state=GLOBAL_SEED)
print(f'  IEEE-CIS: {X_tr_ieee.shape} train, {X_te_ieee.shape} test, fraud={y_te_ieee.mean():.3%}')

# ── H&M Articles ──────────────────────────────────────────────────────────────
print('Loading H&M...', flush=True)
articles = pd.read_csv(DATA_HM / 'articles_hm.csv', low_memory=False)

TOP5 = ['Garment Upper body', 'Garment Lower body',
        'Accessories', 'Socks & Tights', 'Underwear']
articles['is_top_category'] = articles['product_group_name'].isin(TOP5).astype(np.int32)

NUM_HM = ['product_code','product_type_no','graphical_appearance_no',
          'colour_group_code','perceived_colour_value_id',
          'perceived_colour_master_id','department_no',
          'index_group_no','section_no','garment_group_no']
CAT_HM = ['product_type_name','product_group_name','graphical_appearance_name',
          'colour_group_name','perceived_colour_value_name',
          'perceived_colour_master_name','department_name',
          'index_code','index_name','index_group_name',
          'section_name','garment_group_name']

for col in CAT_HM:
    articles[col] = articles[col].fillna('missing')
    articles[col] = LabelEncoder().fit_transform(articles[col]).astype(np.int32)

X_hm = pd.concat([articles[NUM_HM + CAT_HM].fillna(-1).astype(np.float32)] * 3,
                 ignore_index=True)
y_hm = pd.concat([articles['is_top_category']] * 3, ignore_index=True)

X_tr_hm, X_te_hm, y_tr_hm, y_te_hm = train_test_split(
    X_hm, y_hm, test_size=0.2, stratify=y_hm, random_state=GLOBAL_SEED)
print(f'  H&M: {X_tr_hm.shape} train, {X_te_hm.shape} test, top_cat={y_te_hm.mean():.2%}')
print('\n✅ Datasets ready.')


## 5. Model Training
**All models trained with fixed seeds and deterministic settings.**
Same hyperparameters as v2 — documented in full for reproducibility.

In [ ]:
# ── Hyperparameters — fully specified, no defaults hidden ─────────────────────
LGBM_PARAMS = dict(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    n_jobs=1, verbose=-1, random_state=GLOBAL_SEED, class_weight='balanced'
)
XGB_PARAMS = dict(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    early_stopping_rounds=30, n_jobs=1, random_state=GLOBAL_SEED,
    verbosity=0, tree_method='exact'
)
CAT_PARAMS = dict(
    iterations=500, learning_rate=0.05, depth=6,
    random_seed=GLOBAL_SEED, verbose=0, thread_count=1,
    auto_class_weights='Balanced'
)

json.dump(
    {'LightGBM': LGBM_PARAMS, 'XGBoost': XGB_PARAMS, 'CatBoost': CAT_PARAMS},
    open(RESULTS / 'tables' / 'hyperparameters.json', 'w'),
    indent=2, default=str
)


def train_lgbm(X, y):
    m = lgb.LGBMClassifier(**LGBM_PARAMS)
    m.fit(X.astype(np.float64), y,
          eval_set=[(X.astype(np.float64), y)],
          callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    return m


def train_xgb(X, y):
    w = float((y == 0).sum() / (y == 1).sum())
    p = {**XGB_PARAMS, 'scale_pos_weight': w}
    m = xgb.XGBClassifier(**p)
    m.fit(X.astype(np.float64), y,
          eval_set=[(X.astype(np.float64), y)], verbose=False)
    return m


def train_cat(X, y):
    m = cb.CatBoostClassifier(**CAT_PARAMS)
    m.fit(X.astype(np.float64), y)
    return m


print('Training on IEEE-CIS Fraud...')
models_fraud = {}
for name, fn in [('LightGBM', train_lgbm), ('XGBoost', train_xgb), ('CatBoost', train_cat)]:
    print(f'  {name}...', end=' ', flush=True)
    np.random.seed(GLOBAL_SEED)
    m = fn(X_tr_ieee, y_tr_ieee)
    auc = roc_auc_score(y_te_ieee, m.predict_proba(X_te_ieee.astype(np.float64))[:, 1])
    models_fraud[name] = m
    print(f'AUC={auc:.4f}')

print('Training on H&M Ranking...')
models_hm = {}
for name, fn in [('LightGBM', train_lgbm), ('XGBoost', train_xgb), ('CatBoost', train_cat)]:
    print(f'  {name}...', end=' ', flush=True)
    np.random.seed(GLOBAL_SEED)
    m = fn(X_tr_hm, y_tr_hm)
    auc = roc_auc_score(y_te_hm, m.predict_proba(X_te_hm.astype(np.float64))[:, 1])
    models_hm[name] = m
    print(f'AUC={auc:.4f}')

print('\n✅ All models trained deterministically.')


## 6. Inference Pipeline

In [ ]:
class InferencePipeline:
    """
    Three measurable pipeline stages. Precision cast at Stage 1 data entry.
    The GBDT model always receives float64 for predict_proba — this is the
    pipeline-level datatype conversion study, NOT internal model precision
    modification. Explicitly disclosed per reviewer feedback.
    On this CPU (Intel Xeon Broadwell, f16c only):
      FP32 maps to native 256-bit AVX2 lanes (8 values per cycle).
      FP16/BF16 require VCVTPH2PS widening before arithmetic — higher cost.
    """
    def __init__(self, model, name, X_ref):
        self.model  = model
        self.name   = name
        self.scaler = StandardScaler()
        self.scaler.fit(X_ref.astype(np.float64))

    def preprocess(self, X, precision):
        """Stage 1 (P_pre): cast -> NaN fill -> scale -> clip."""
        dtype = PRECISION_PROFILES[precision]
        arr   = X.values.astype(dtype)
        arr   = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        # Cast scaler params to target dtype to stay in precision profile
        mean_ = self.scaler.mean_.astype(dtype)
        std_  = self.scaler.scale_.astype(dtype)
        # Use float literal for epsilon — avoids dtype(type(x)(x)) bug
        arr   = (arr - mean_) / (std_ + np.float32(1e-7).astype(dtype))
        arr   = np.clip(arr, -10.0, 10.0)
        return arr.astype(dtype)

    def infer(self, X_pre):
        """Stage 2 (M_inf): GBDT predict_proba. Widened to float64."""
        return self.model.predict_proba(X_pre.astype(np.float64))[:, 1]

    def postprocess(self, scores, task):
        """Stage 3 (P_post): business-logic output translation."""
        if task == 'fraud':
            decisions  = (scores >= 0.5).astype(np.int32)
            confidence = np.abs(scores - 0.5)
            return np.stack([decisions, confidence], axis=1)
        s    = scores
        norm = (s - s.min()) / (s.max() - s.min() + 1e-8)
        return norm[np.argsort(-norm)]


pipes_fraud = {n: InferencePipeline(m, n, X_tr_ieee) for n, m in models_fraud.items()}
pipes_hm    = {n: InferencePipeline(m, n, X_tr_hm)   for n, m in models_hm.items()}
print('✅ Pipelines ready.')


## 7. Experiment Loop
⏱️ **~5-6 hours on free Colab CPU.**
Results checkpoint-saved after each dataset. If session dies, reload from checkpoint and run only Cell 7b.

**Run Cell 7a first (fraud), then Cell 7b (H&M). Do not run both in sequence unattended.**

In [ ]:
# ── EXPERIMENT FUNCTION ───────────────────────────────────────────────────────
def run_experiment(pipes, X_ref, y_ref, task, dataset_name):
    records = []
    n_prec  = len(PRECISION_PROFILES)
    total   = len(pipes) * n_prec * len(BATCH_SIZES)
    done    = 0

    for model_name, pipe in pipes.items():

        # FP64 baseline scores for drift (fixed reference, fixed seed)
        np.random.seed(GLOBAL_SEED)
        ref_idx    = np.random.choice(len(X_ref), min(500, len(X_ref)), replace=False)
        X_ref_pre  = pipe.preprocess(X_ref.iloc[ref_idx], 'FP64')
        base_scores = pipe.infer(X_ref_pre)

        for precision in PRECISION_PROFILES:
            for batch_size in BATCH_SIZES:
                done += 1
                print(f'  [{done:>3}/{total}] {dataset_name:<20} | {model_name:<10} | '
                      f'{precision:<5} | batch={batch_size:>7,}', flush=True)

                # Fixed-seed batch sampling — same batch every repeat for consistency
                np.random.seed(GLOBAL_SEED + done)  # deterministic per config
                n = len(X_ref)
                idx = np.random.choice(n, batch_size, replace=(batch_size > n))
                X_b = X_ref.iloc[idx].reset_index(drop=True)
                y_b = y_ref.iloc[idx].reset_index(drop=True)

                # Per-repeat buffers
                buf = {s: {'lat':[], 'en':[], 'co2':[]}
                       for s in ['preprocessing','inference','postprocessing']}
                auc_buf   = []
                drift_buf = []

                for rep in range(N_REPEATS + N_WARMUP):

                    with measure_stage('preprocessing', precision,
                                       batch_size, model_name, dataset_name) as m1:
                        X_proc = pipe.preprocess(X_b, precision)

                    with measure_stage('inference', precision,
                                       batch_size, model_name, dataset_name) as m2:
                        scores = pipe.infer(X_proc)

                    with measure_stage('postprocessing', precision,
                                       batch_size, model_name, dataset_name) as m3:
                        _ = pipe.postprocess(scores, task)

                    if rep < N_WARMUP:
                        continue  # discard warm-up runs

                    for s_name, mm in [('preprocessing',m1),('inference',m2),('postprocessing',m3)]:
                        buf[s_name]['lat'].append(mm.latency_ms)
                        buf[s_name]['en'].append(mm.energy_mj)
                        buf[s_name]['co2'].append(mm.co2e_ug)

                    # AUC — same test batch, fixed seed
                    try:
                        auc_buf.append(roc_auc_score(y_b, scores))
                    except Exception:
                        auc_buf.append(np.nan)

                    # Score drift vs FP64 baseline
                    n_cmp  = min(500, batch_size)
                    s_curr = pipe.infer(pipe.preprocess(X_b.iloc[:n_cmp], precision))
                    s_base = pipe.infer(pipe.preprocess(X_b.iloc[:n_cmp], 'FP64'))
                    drift_buf.append(float(np.mean(np.abs(s_curr - s_base))))

                # ── Aggregate ─────────────────────────────────────────────────
                for s_name, vals in buf.items():
                    lat_arr = np.array(vals['lat'])
                    en_arr  = np.array(vals['en'])
                    co2_arr = np.array(vals['co2'])

                    # 95% CI via t-distribution (n=25)
                    def ci95(arr):
                        n = len(arr)
                        se = arr.std(ddof=1) / np.sqrt(n)
                        t  = stats.t.ppf(0.975, df=n-1)
                        return float(se * t)

                    records.append({
                        'dataset'           : dataset_name,
                        'model'             : model_name,
                        'precision'         : precision,
                        'batch_size'        : batch_size,
                        'stage'             : s_name,
                        'latency_ms_mean'   : float(lat_arr.mean()),
                        'latency_ms_std'    : float(lat_arr.std(ddof=1)),
                        'latency_ms_ci95'   : ci95(lat_arr),
                        'latency_ms_p99'    : float(np.percentile(lat_arr, 99)),
                        'energy_mj_mean'    : float(en_arr.mean()),
                        'energy_mj_std'     : float(en_arr.std(ddof=1)),
                        'energy_mj_ci95'    : ci95(en_arr),
                        'energy_mj_all'     : en_arr.tolist(),  # keep raw for Wilcoxon
                        'co2e_ug_mean'      : float(co2_arr.mean()),
                        'auc_mean'          : float(np.nanmean(auc_buf)),
                        'auc_std'           : float(np.nanstd(auc_buf)),
                        'score_drift_mean'  : float(np.mean(drift_buf)),
                        'score_drift_std'   : float(np.std(drift_buf)),
                        'n_repeats'         : N_REPEATS,
                    })

    return pd.DataFrame(records)

print('✅ Experiment function ready.')

In [ ]:
# ── CELL 7a — FRAUD DATASET (~2.5-3 hours) ────────────────────────────────────
# Run this cell, wait for it to finish, then run Cell 7b.
print('='*65)
print('RUN 1: IEEE-CIS Fraud Detection')
print('='*65)
df_fraud = run_experiment(pipes_fraud, X_te_ieee, y_te_ieee,
                          'fraud', 'IEEE-CIS Fraud')

# Save immediately — checkpoint
df_fraud.to_csv(RESULTS/'tables'/'results_fraud_v3.csv', index=False)
# Also save raw energy lists separately for Wilcoxon
import json
raw_fraud = df_fraud[['dataset','model','precision','batch_size','stage','energy_mj_all']].copy()
raw_fraud.to_json(RESULTS/'tables'/'raw_energy_fraud_v3.json', orient='records')

print(f'\n✅ CHECKPOINT: Fraud results saved. Rows: {len(df_fraud)}')
print('Now run Cell 7b for H&M dataset.')

In [ ]:
# ── CELL 7b — H&M DATASET (~2.5-3 hours) ─────────────────────────────────────
print('='*65)
print('RUN 2: H&M Product Candidate Generation')
print('='*65)
df_hm = run_experiment(pipes_hm, X_te_hm, y_te_hm,
                       'ranking', 'HM Ranking')

# Save immediately — checkpoint
df_hm.to_csv(RESULTS/'tables'/'results_hm_v3.csv', index=False)
raw_hm = df_hm[['dataset','model','precision','batch_size','stage','energy_mj_all']].copy()
raw_hm.to_json(RESULTS/'tables'/'raw_energy_hm_v3.json', orient='records')

# Combine
df_all = pd.concat([df_fraud, df_hm], ignore_index=True)
df_all.to_csv(RESULTS/'tables'/'results_all_v3.csv', index=False)

print(f'\n✅ CHECKPOINT: H&M results saved. Total rows: {len(df_all)}')

## 8. Statistical Analysis
Run after both dataset cells complete. Computes Wilcoxon tests, effect sizes, and p-values.

In [ ]:
# ── Reload if needed ──────────────────────────────────────────────────────────
import ast
df_all = pd.read_csv(RESULTS/'tables'/'results_all_v3.csv')

# Reload raw energy lists from JSON for Wilcoxon
raw_f = pd.read_json(RESULTS/'tables'/'raw_energy_fraud_v3.json')
raw_h = pd.read_json(RESULTS/'tables'/'raw_energy_hm_v3.json')
raw   = pd.concat([raw_f, raw_h], ignore_index=True)


def get_energy_array(df_raw, dataset, model, precision, stage, batch_size):
    """Extract raw energy list for a given config."""
    row = df_raw[
        (df_raw.dataset    == dataset)   &
        (df_raw.model      == model)     &
        (df_raw.precision  == precision) &
        (df_raw.batch_size == batch_size)&
        (df_raw.stage      == stage)
    ]
    if row.empty:
        return np.array([])
    val = row.iloc[0]['energy_mj_all']
    if isinstance(val, str):
        val = ast.literal_eval(val)
    return np.array(val)


# ── Wilcoxon signed-rank: FP32 vs FP64, FP16 vs FP64, BF16 vs FP64 ──────────
stat_rows = []
comparisons = [('FP32','FP64'), ('FP16','FP64')]
if 'BF16' in df_all.precision.unique():
    comparisons.append(('BF16','FP64'))

for dataset in df_all.dataset.unique():
    for model in df_all.model.unique():
        for stage in ['preprocessing','inference','postprocessing']:
            for batch_size in BATCH_SIZES:
                for prec_a, prec_b in comparisons:
                    arr_a = get_energy_array(raw, dataset, model, prec_a, stage, batch_size)
                    arr_b = get_energy_array(raw, dataset, model, prec_b, stage, batch_size)

                    if len(arr_a) < 5 or len(arr_b) < 5:
                        continue

                    try:
                        stat, p = wilcoxon(arr_a, arr_b, alternative='two-sided')
                        # Effect size: rank-biserial correlation
                        n = len(arr_a)
                        r = 1 - (2*stat) / (n*(n+1)/2)
                    except Exception:
                        p, r = np.nan, np.nan

                    mean_a = arr_a.mean()
                    mean_b = arr_b.mean()
                    saving_pct = (mean_b - mean_a) / mean_b * 100 if mean_b > 0 else 0

                    stat_rows.append({
                        'dataset'     : dataset,
                        'model'       : model,
                        'stage'       : stage,
                        'batch_size'  : batch_size,
                        'comparison'  : f'{prec_a} vs {prec_b}',
                        'mean_a_mj'   : round(mean_a, 4),
                        'mean_b_mj'   : round(mean_b, 4),
                        'saving_pct'  : round(saving_pct, 2),
                        'wilcoxon_p'  : round(p, 5) if not np.isnan(p) else 'n/a',
                        'effect_size_r': round(r, 3) if not np.isnan(r) else 'n/a',
                        'significant' : 'YES' if (not np.isnan(p) and p < 0.05) else 'NO',
                    })

df_stats = pd.DataFrame(stat_rows)
df_stats.to_csv(RESULTS/'stats'/'wilcoxon_results.csv', index=False)

# Print summary
print('=== WILCOXON SIGNIFICANCE SUMMARY ===')
print(df_stats.groupby(['comparison','significant']).size().unstack(fill_value=0))
print()
print('=== FP32 vs FP64 — ALL SIGNIFICANT? ===')
fp32_sig = df_stats[df_stats.comparison=='FP32 vs FP64']
print(f'  Significant (p<0.05): {(fp32_sig.significant=="YES").sum()}/{len(fp32_sig)}')
print(f'  Mean energy saving  : {fp32_sig.saving_pct.mean():.1f}%')
print()
print('=== FP16 vs FP64 — OVERHEAD CONFIRMED? ===')
fp16_sig = df_stats[df_stats.comparison=='FP16 vs FP64']
fp16_overhead = fp16_sig[fp16_sig.saving_pct < 0]
print(f'  Configs with overhead: {len(fp16_overhead)}/{len(fp16_sig)}')
print(f'  Mean overhead: {-fp16_sig.saving_pct.mean():.1f}%')
print()
print('✅ Statistical analysis complete.')

## 9. Figures with Error Bars
All figures now include 95% CI error bars — directly addresses reviewer criticism.

In [ ]:
df_all = pd.read_csv(RESULTS/'tables'/'results_all_v3.csv')

# Determine precision list from data
PREC_LIST   = [p for p in ['FP64','FP32','FP16','BF16'] if p in df_all.precision.unique()]
COLORS      = {'FP64':'#2166AC','FP32':'#1A9641','FP16':'#D73027','BF16':'#7B2D8B'}
MARKERS     = {'FP64':'o','FP32':'s','FP16':'^','BF16':'D'}
STAGES      = ['preprocessing','inference','postprocessing']
MODELS      = ['LightGBM','XGBoost','CatBoost']
plt.rcParams.update({'figure.dpi':150,'font.size':10,'font.family':'DejaVu Sans'})

print(f'Precisions in data: {PREC_LIST}')
print('✅ Plot config ready.')

In [ ]:
# ── FIGURE 1: Stage-Wise Energy Decomposition WITH error bars ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Fig 1: Stage-Wise Energy Decomposition by Precision\n'
    '(LightGBM, Batch=10,000, error bars = 95% CI, n=25)',
    fontsize=12, fontweight='bold'
)

for ax, dataset in zip(axes, ['IEEE-CIS Fraud', 'HM Ranking']):
    # Simple equality filter — no ternary inside boolean mask
    sub = df_all[
        (df_all['dataset']     == dataset) &
        (df_all['model']       == 'LightGBM') &
        (df_all['batch_size']  == 10_000)
    ]

    piv_mean = sub.groupby(['stage', 'precision'])['energy_mj_mean'].mean().unstack('precision')
    piv_ci   = sub.groupby(['stage', 'precision'])['energy_mj_ci95'].mean().unstack('precision')
    piv_mean = piv_mean.reindex([s for s in STAGES if s in piv_mean.index])
    piv_ci   = piv_ci.reindex(piv_mean.index)

    x = np.arange(len(piv_mean))
    w = 0.8 / len(PREC_LIST)

    for i, prec in enumerate(PREC_LIST):
        if prec not in piv_mean.columns:
            continue
        offset = (i - len(PREC_LIST) / 2 + 0.5) * w
        ax.bar(
            x + offset, piv_mean[prec], w,
            label=prec, color=COLORS[prec], alpha=0.85, edgecolor='white',
            yerr=piv_ci[prec], capsize=4, error_kw={'elinewidth': 1.2}
        )

    ax.set_title(dataset, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([s.capitalize() for s in piv_mean.index])
    ax.set_xlabel('Pipeline Stage')
    ax.set_ylabel('Energy (mJ)')
    ax.legend(title='Precision')
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(RESULTS / 'figures' / 'fig1_stage_energy_v3.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Fig 1 saved.')


In [ ]:
# ── FIGURE 2: Energy vs Scale WITH confidence bands ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Fig 2: Total Pipeline Energy vs Inference Scale\n'
             '(IEEE-CIS Fraud, shaded band = 95% CI, n=25)',
             fontsize=12, fontweight='bold')

for ax, model in zip(axes, MODELS):
    sub = df_all[(df_all.dataset=='IEEE-CIS Fraud') & (df_all.model==model)]
    for prec in PREC_LIST:
        ps = sub[sub.precision==prec].groupby('batch_size').agg(
            mean=('energy_mj_mean','sum'),
            ci=('energy_mj_ci95', lambda x: x.sum())
        )
        ax.plot(ps.index, ps['mean'], marker=MARKERS[prec],
                color=COLORS[prec], linewidth=2, label=prec)
        ax.fill_between(ps.index,
                        ps['mean'] - ps['ci'],
                        ps['mean'] + ps['ci'],
                        color=COLORS[prec], alpha=0.15)

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_title(model, fontweight='bold')
    ax.set_xlabel('Batch Size (log)'); ax.set_ylabel('Total Energy (mJ, log)')
    ax.legend(title='Precision'); ax.grid(True, alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(RESULTS/'figures'/'fig2_energy_scale_v3.png', dpi=300, bbox_inches='tight')
plt.show(); print('✅ Fig 2 saved.')

In [ ]:
# ── FIGURE 3: AUC vs Precision WITH std error bars ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Fig 3: AUC-ROC vs Precision Profile\n'
             '(error bars = ±1 std across 25 repeats)',
             fontsize=12, fontweight='bold')

for ax, dataset in zip(axes, ['IEEE-CIS Fraud','HM Ranking']):
    sub = df_all[
        (df_all.dataset==dataset) &
        (df_all.stage=='inference') &
        (df_all.batch_size==10_000)
    ].groupby(['model','precision']).agg(
        auc=('auc_mean','mean'),
        auc_std=('auc_std','mean')
    ).unstack('precision')

    x = np.arange(len(sub))
    w = 0.8 / len(PREC_LIST)
    for i, prec in enumerate(PREC_LIST):
        if prec not in sub['auc'].columns: continue
        offset = (i - len(PREC_LIST)/2 + 0.5) * w
        ax.bar(x + offset, sub['auc'][prec], w,
               label=prec, color=COLORS[prec], alpha=0.85, edgecolor='white',
               yerr=sub['auc_std'][prec], capsize=4,
               error_kw={'elinewidth':1.2})

    ax.set_ylim(0.5, 1.02)
    ax.axhline(0.95, color='gray', linestyle='--', alpha=0.5, label='0.95 ref')
    ax.set_title(dataset, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(sub.index, rotation=0)
    ax.set_xlabel('Model'); ax.set_ylabel('AUC-ROC')
    ax.legend(title='Precision'); ax.grid(axis='y', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(RESULTS/'figures'/'fig3_auc_v3.png', dpi=300, bbox_inches='tight')
plt.show(); print('✅ Fig 3 saved.')

In [ ]:
# ── FIGURE 4: Score Drift Heatmap ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Fig 4: Score Drift Heatmap (vs FP64 Baseline)\n'
             '(LightGBM, Batch=10,000, mean over 25 repeats)',
             fontsize=12, fontweight='bold')

for ax, dataset in zip(axes, ['IEEE-CIS Fraud','HM Ranking']):
    pivot = df_all[
        (df_all.dataset==dataset) & (df_all.model=='LightGBM') &
        (df_all.batch_size==10_000)
    ].groupby(['stage','precision'])['score_drift_mean'].mean().unstack('precision')
    pivot = pivot.reindex([s for s in STAGES if s in pivot.index])
    pivot.index = [s.capitalize() for s in pivot.index]

    sns.heatmap(pivot[PREC_LIST], ax=ax, annot=True, fmt='.5f',
                cmap='RdYlGn_r', linewidths=0.5,
                cbar_kws={'label':'Mean Abs Score Drift'})
    ax.set_title(dataset, fontweight='bold')
    ax.set_xlabel('Precision'); ax.set_ylabel('Pipeline Stage')

plt.tight_layout()
plt.savefig(RESULTS/'figures'/'fig4_drift_v3.png', dpi=300, bbox_inches='tight')
plt.show(); print('✅ Fig 4 saved.')

In [ ]:
# ── FIGURE 5: P99 Latency ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Fig 5: P99 Latency vs Batch Size\n'
             '(Inference Stage, n=25 repeats)',
             fontsize=12, fontweight='bold')

for ax, dataset in zip(axes, ['IEEE-CIS Fraud','HM Ranking']):
    sub = df_all[
        (df_all.dataset==dataset) & (df_all.model=='LightGBM') &
        (df_all.stage=='inference')
    ].groupby(['batch_size','precision'])['latency_ms_p99'].mean().unstack('precision')

    for prec in PREC_LIST:
        if prec in sub.columns:
            ax.plot(sub.index, sub[prec], marker=MARKERS[prec],
                    color=COLORS[prec], linewidth=2, label=prec)

    ax.set_xscale('log')
    ax.set_title(dataset, fontweight='bold')
    ax.set_xlabel('Batch Size (log)'); ax.set_ylabel('P99 Latency (ms)')
    ax.legend(title='Precision'); ax.grid(True, alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(RESULTS/'figures'/'fig5_latency_v3.png', dpi=300, bbox_inches='tight')
plt.show(); print('✅ Fig 5 saved.')

In [ ]:
# ── FIGURE 6: FP16/BF16 Overhead Decomposition ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Fig 6: Precision Energy Overhead Decomposed by Stage\n'
             '(IEEE-CIS Fraud, Batch=10,000, % vs FP64, error bars = 95% CI)',
             fontsize=12, fontweight='bold')

compare_precs = [p for p in ['FP32','FP16','BF16'] if p in PREC_LIST]
COMP_COLORS   = {'FP32':'#1A9641','FP16':'#D73027','BF16':'#7B2D8B'}

for ax, model in zip(axes, MODELS):
    sub = df_all[
        (df_all.dataset=='IEEE-CIS Fraud') &
        (df_all.model==model) &
        (df_all.batch_size==10_000)
    ].groupby(['stage','precision']).agg(
        mean=('energy_mj_mean','mean'),
        ci=('energy_mj_ci95','mean')
    ).unstack('precision')

    stages_avail = [s for s in STAGES if s in sub.index]
    x = np.arange(len(stages_avail))
    w = 0.75 / len(compare_precs)

    for i, prec in enumerate(compare_precs):
        if prec not in sub['mean'].columns: continue
        fp64 = sub['mean']['FP64']
        pct  = (sub['mean'][prec] / fp64 - 1) * 100
        ci_v = sub['ci'][prec]
        offset = (i - len(compare_precs)/2 + 0.5) * w
        bars = ax.bar(x + offset, pct, w, label=prec,
                      color=COMP_COLORS[prec], alpha=0.85,
                      edgecolor='white', capsize=4)
        # Annotate percentage
        for xi, (p, b) in enumerate(zip(pct, bars)):
            sign = '+' if p >= 0 else ''
            ax.text(b.get_x() + b.get_width()/2,
                    p + (2 if p >= 0 else -4),
                    f'{sign}{p:.0f}%',
                    ha='center', fontsize=7, fontweight='bold',
                    color=COMP_COLORS[prec])

    ax.axhline(0, color='black', lw=1)
    ax.set_title(model, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Preprocessing','Inference','Post-proc.'], fontsize=9)
    ax.set_xlabel('Pipeline Stage')
    ax.set_ylabel('Energy change vs FP64 (%)')
    ax.legend(title='Precision'); ax.grid(axis='y', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(RESULTS/'figures'/'fig6_overhead_v3.png', dpi=300, bbox_inches='tight')
plt.show(); print('✅ Fig 6 saved.')

## 10. Summary Tables

In [ ]:
# ── TABLE 1: Energy summary with CI ──────────────────────────────────────────
summary = df_all.groupby(['dataset','model','precision']).agg(
    total_energy_mj   = ('energy_mj_mean','sum'),
    mean_ci95_mj      = ('energy_mj_ci95','mean'),
    mean_latency_ms   = ('latency_ms_mean','mean'),
    p99_latency_ms    = ('latency_ms_p99','mean'),
    mean_auc          = ('auc_mean','mean'),
    mean_auc_std      = ('auc_std','mean'),
    mean_score_drift  = ('score_drift_mean','mean'),
).round(4).reset_index()

# Energy saving % vs FP64
base = summary[summary.precision=='FP64'][['dataset','model','total_energy_mj']]
base = base.rename(columns={'total_energy_mj':'base_mj'})
summary = summary.merge(base, on=['dataset','model'])
summary['energy_saving_pct'] = ((1 - summary.total_energy_mj / summary.base_mj)*100).round(2)

# Energy per 1000 inferences (normalised — reviewer request)
# Use batch=10000 total pipeline energy / 10000 * 1000
per1k = df_all[
    df_all.batch_size==10_000
].groupby(['dataset','model','precision'])['energy_mj_mean'].sum().reset_index()
per1k['energy_per_1k_mj'] = (per1k['energy_mj_mean'] / 10_000 * 1_000).round(4)
per1k = per1k[['dataset','model','precision','energy_per_1k_mj']]
summary = summary.merge(per1k, on=['dataset','model','precision'])

summary.to_csv(RESULTS/'tables'/'summary_table_v3.csv', index=False)
print('TABLE 1: Energy & Accuracy Summary')
print(summary[['dataset','model','precision','total_energy_mj','mean_ci95_mj',
               'energy_saving_pct','energy_per_1k_mj','mean_auc','mean_auc_std'
               ]].to_string(index=False))
print('\n✅ Summary table saved.')

In [ ]:
# ── CARBON PROJECTIONS (recalculated carefully) ───────────────────────────────
# Get per-prediction energy from batch=10000, inference stage, LightGBM fraud
ref = df_all[
    (df_all.dataset=='IEEE-CIS Fraud') &
    (df_all.model=='LightGBM') &
    (df_all.batch_size==10_000)
].groupby('precision')['energy_mj_mean'].sum()

per_pred_mj = ref / 10_000   # mJ per prediction

print('Per-prediction energy (LightGBM, Fraud, batch=10k):')
for p, v in per_pred_mj.items():
    print(f'  {p}: {v:.6f} mJ/prediction  = {v*1e-9:.6f} GJ/prediction')

print()
DAILY_INF = {'Amazon':1_000_000_000_000, 'Alibaba':864_000_000_000}

proj_rows = []
for company, daily in DAILY_INF.items():
    fp64_mj_day = per_pred_mj.get('FP64', 0) * daily
    fp64_gj_day = fp64_mj_day * 1e-9
    fp64_kwh    = fp64_mj_day / 3.6e9

    for prec in PREC_LIST:
        if prec not in per_pred_mj.index: continue
        daily_mj  = per_pred_mj[prec] * daily
        daily_gj  = daily_mj  * 1e-9
        daily_kwh = daily_mj  / 3.6e9
        saving_gj = fp64_gj_day - daily_gj
        saving_pct= saving_gj / fp64_gj_day * 100 if fp64_gj_day > 0 else 0

        proj_rows.append({
            'Company'         : company,
            'Daily_Inferences': f'{daily:,}',
            'Precision'       : prec,
            'per_pred_mj'     : round(per_pred_mj[prec], 8),
            'Daily_Energy_GJ' : round(daily_gj, 3),
            'Daily_Energy_kWh': round(daily_kwh, 1),
            'Saving_vs_FP64_GJ': round(saving_gj, 3),
            'Saving_pct'      : round(saving_pct, 2),
        })

df_proj = pd.DataFrame(proj_rows)
df_proj.to_csv(RESULTS/'tables'/'carbon_projections_v3.csv', index=False)
print(df_proj.to_string(index=False))
print('\n✅ Carbon projections saved (with full unit breakdown for verification).')

In [ ]:
# ── FINAL FILE LIST ───────────────────────────────────────────────────────────
import os
print('ALL FILES SAVED TO DRIVE (results_v3/):')
for p in sorted(RESULTS.rglob('*')):
    if p.is_file():
        rel = str(p).replace(str(BASE)+'/', '')
        print(f'  {rel}  ({p.stat().st_size//1024} KB)')